# آزمایشگاه کلاسی: Embedding با API در OpenRouter

۱۶۰ پست فارسی تلگرام را با یک مدل embedding که از طریق API فراخوانی می‌شود تحلیل می‌کنیم. در این تمرین gold standard نداریم؛ خروجی یک نقشه اولیه برای بازبینی پژوهشگر است.

## پیش‌نیاز

1. در [OpenRouter](https://openrouter.ai/) حساب بسازید و از بخش **Keys** یک API key ایجاد کنید.
2. در [Google Colab](https://colab.research.google.com/) از **File → Upload notebook** این فایل را باز کنید.
3. کلید را فقط در کادر مخفی وارد کنید؛ آن را در کد یا اسکرین‌شات نشان ندهید.

In [ ]:
# نصب — فقط یک‌بار اجرا کنید
!pip -q install pandas scikit-learn matplotlib requests

In [ ]:
# آپلود فایل داده (فایل را از Student Pack انتخاب کنید)
from google.colab import files
uploaded = files.upload()

In [ ]:
import io
from getpass import getpass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.decomposition import PCA

DATA_FILE = 'telegram_policy_radar_4topics_fa.csv'
df = pd.read_csv(io.BytesIO(uploaded[DATA_FILE]))
print(f'تعداد پست‌ها: {len(df)}')
df.head(3)

## چهار موضوع راهنما

مدل برای هر پست و هر توضیح موضوع یک بردار می‌سازد. سپس cosine similarity نزدیک‌ترین موضوع را انتخاب می‌کند. توضیح‌ها را تغییر دهید و اثرش را ببینید.

In [ ]:
TOPICS = {
    'تنظیم‌گری و سیاست‌گذاری': 'اخبار مربوط به قانون‌گذاری، مجوز، مقررات، نهادهای دولتی، شوراها و سیاست‌گذاری اقتصاد دیجیتال.',
    'هوش مصنوعی و داده': 'اخبار مربوط به هوش مصنوعی، مدل‌های زبانی، داده، الگوریتم‌ها و کاربردهای هوش مصنوعی.',
    'رقابت و پلتفرم‌های دیجیتال': 'اخبار مربوط به رقابت، انحصار، قدرت بازار، پلتفرم‌ها، تجارت الکترونیک و بازیگران دیجیتال.',
    'زیرساخت و مخابرات': 'اخبار مربوط به اینترنت، اپراتورها، فیبر نوری، شبکه، 5G و زیرساخت ارتباطی.'
}
CODES = {'تنظیم‌گری و سیاست‌گذاری': 'REG', 'هوش مصنوعی و داده': 'AI', 'رقابت و پلتفرم‌های دیجیتال': 'COMP', 'زیرساخت و مخابرات': 'TELCO'}
EMBEDDING_MODEL = 'openai/text-embedding-3-small'
TOPICS

## اتصال امن به API

هر دانشجو باید کلید خودش را وارد کند. کلید در فایل یا خروجی ذخیره نمی‌شود.

In [ ]:
OPENROUTER_API_KEY = getpass('OpenRouter API key (hidden): ')
assert OPENROUTER_API_KEY.strip(), 'یک API key لازم است.'
HEADERS = {'Authorization': f'Bearer {OPENROUTER_API_KEY}', 'Content-Type': 'application/json', 'HTTP-Referer': 'https://colab.research.google.com/', 'X-OpenRouter-Title': 'BANA Embeddings Workshop'}
API_URL = 'https://openrouter.ai/api/v1/embeddings'

## ارسال درخواست embedding به API

متن‌ها در batchهای ۳۲تایی فرستاده می‌شوند. پاسخ API شامل بردار embedding و تعداد token مصرف‌شده است.

In [ ]:
def embed_batch(texts):
    response = requests.post(API_URL, headers=HEADERS, json={'model': EMBEDDING_MODEL, 'input': texts, 'encoding_format': 'float'}, timeout=90)
    response.raise_for_status()
    payload = response.json()
    vectors = [item['embedding'] for item in sorted(payload['data'], key=lambda item: item['index'])]
    return vectors, payload.get('usage', {}).get('total_tokens', 0)

def embed_all(texts, batch_size=32):
    vectors, total_tokens = [], 0
    for start in range(0, len(texts), batch_size):
        batch_vectors, batch_tokens = embed_batch(texts[start:start + batch_size])
        vectors.extend(batch_vectors)
        total_tokens += batch_tokens
        print(f'پردازش شد: {min(start + batch_size, len(texts))}/{len(texts)}')
    return np.asarray(vectors, dtype='float32'), total_tokens

## ساخت embedding و طبقه‌بندی

API برای پست‌ها و چهار توضیح موضوع فراخوانی می‌شود. بعد از اجرا، تعداد token مصرف‌شده را ببینید.

In [ ]:
post_vectors, post_tokens = embed_all(df['text_fa'].tolist())
topic_vectors, topic_tokens = embed_all(list(TOPICS.values()))
post_vectors /= np.linalg.norm(post_vectors, axis=1, keepdims=True)
topic_vectors /= np.linalg.norm(topic_vectors, axis=1, keepdims=True)
scores = post_vectors @ topic_vectors.T
labels = list(TOPICS)
df['predicted_topic'] = [labels[i] for i in scores.argmax(axis=1)]
df['similarity'] = scores.max(axis=1).round(3)
df['review_needed'] = np.where(df['similarity'] < 0.40, 'yes', 'no')
print('Model:', EMBEDDING_MODEL, '| API tokens:', post_tokens + topic_tokens)
df[['student_id', 'predicted_topic', 'similarity', 'review_needed', 'text_fa']].head()

## ۱) توزیع موضوع‌ها

این نمودار جواب می‌دهد: در این نمونه، کدام حوزه‌ها بیشترین حضور را دارند؟

In [ ]:
counts = df['predicted_topic'].value_counts().reindex(labels, fill_value=0)
plt.figure(figsize=(8, 4.5))
plt.bar([CODES[x] for x in labels], counts.values, color=['#5B8FF9', '#61DDAA', '#65789B', '#F6BD16'])
plt.title('Telegram policy radar: predicted themes')
plt.ylabel('Posts')
for i, value in enumerate(counts.values):
    plt.text(i, value + 1, str(value), ha='center')
plt.show()
pd.DataFrame({'topic': labels, 'code': [CODES[x] for x in labels], 'posts': counts.values})

## ۲) نقشه embedding

هر نقطه یک پست است. فاصله کمتر یعنی شباهت معنایی بیشتر؛ این نمایش دو‌بعدی، نسخه فشرده‌شده‌ای از بردارهای چندصدبعدی است.

In [ ]:
coords = PCA(n_components=2, random_state=42).fit_transform(post_vectors)
plt.figure(figsize=(8.5, 6.5))
for code, color, label in zip(['REG', 'AI', 'COMP', 'TELCO'], ['#5B8FF9', '#61DDAA', '#65789B', '#F6BD16'], labels):
    mask = df['predicted_topic'].eq(label)
    plt.scatter(coords[mask, 0], coords[mask, 1], label=code, color=color, alpha=.72, s=34)
plt.title('Telegram policy radar: embedding map (PCA)')
plt.xlabel('PCA dimension 1'); plt.ylabel('PCA dimension 2')
plt.legend(title='Topic code')
plt.show()

## ۳) نماینده‌ترین پست‌ها در هر موضوع

شباهت بالاتر الزاماً «درست‌تر» نیست؛ فقط یعنی پست به توضیح آن موضوع نزدیک‌تر است.

In [ ]:
for label in labels:
    print('\n' + '=' * 70)
    print(CODES[label], '—', label)
    display(df[df['predicted_topic'].eq(label)].nlargest(3, 'similarity')[['similarity', 'text_fa']])

## ۴) جست‌وجوی معنایی

جمله جست‌وجو را تغییر دهید، سپس این سلول را دوباره اجرا کنید. این روش برای مرور سریع یک پیکره بزرگ مفید است.

In [ ]:
QUERY = 'خبرهایی درباره مقررات و سیاست‌گذاری پلتفرم‌های دیجیتال'
query_vectors, query_tokens = embed_all([QUERY])
query_vector = query_vectors[0] / np.linalg.norm(query_vectors[0])
df['query_similarity'] = post_vectors @ query_vector
print('Query:', QUERY, '| API tokens:', query_tokens)
display(df.nlargest(5, 'query_similarity')[['query_similarity', 'predicted_topic', 'text_fa']])

## ۵) موارد نیازمند بازبینی انسانی

این‌ها پست‌هایی‌اند که حتی بهترین موضوع برایشان شباهت نسبتاً کمی داشته است. آیا می‌توانید دلیل ابهام را توضیح دهید؟

In [ ]:
review_queue = df.nsmallest(12, 'similarity')[['student_id', 'predicted_topic', 'similarity', 'text_fa']]
display(review_queue)

## ۶) دریافت نتایج و بحث کلاسی

پاسخ کوتاه خود را آماده کنید:

- دو موضوع غالب کدام‌اند و چه توضیحی برای آن دارید؟
- دو پست نماینده چه الگوی مشترکی دارند؟
- یک مورد کم‌اطمینان را بخوانید: چرا طبقه‌بندی آن دشوار است؟
- اگر این کار برای یک گزارش واقعی BANA بود، چه چیزی را پیش از تصمیم‌گیری تغییر یا بازبینی می‌کردید؟

In [ ]:
# ذخیره و دانلود نتایج
OUTPUT_FILE = 'telegram_policy_radar_4topics_openrouter_results.csv'
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
from google.colab import files
files.download(OUTPUT_FILE)